# [M1] NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
## Buổi S01: Python cho Data & Mảng Đa Chiều với NumPy

---

### Metadata Khóa Học
* **Module:** M1 — NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
* **Buổi:** S01
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook tối ưu hóa tính toán số học bằng NumPy Vectorization
* **Mức độ liên quan VOAI / Olympic AI:** Xử lý nhanh dữ liệu mảng, chuẩn bị đặc trưng không nghẽn cổ chai thời gian.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu sự khác biệt về cấu trúc bộ nhớ và tốc độ giữa Python List và NumPy ndarray.
- Làm chủ kỹ thuật Vectorization (vectơ hóa), loại bỏ hoàn toàn các vòng lặp for chậm chạp trong xử lý dữ liệu số.
- Nắm vững cơ chế Broadcasting (lan truyền chiều) trong các phép tính toán ma trận.
- Thực hành tính toán thống kê và trích xuất dữ liệu đa chiều bằng Boolean Masking.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Module M0.
- Biết cú pháp cơ bản của Python (biến, vòng lặp, hàm).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong thi đấu Olympic AI, bạn thường xuyên phải làm việc với các bảng dữ liệu hàng triệu dòng hoặc tensor hình ảnh có hàng chục triệu pixel. Nếu xử lý bằng vòng lặp `for` thuần trong Python (Python loop), thời gian thực thi có thể mất hàng chục phút, dẫn tới Time Limit Exceeded (TLE).

NumPy là "trái tim" tính toán số học của toàn bộ hệ sinh thái AI (Scikit-Learn, PyTorch, TensorFlow đều xây dựng trên nền tảng ý tưởng của NumPy). Bằng cách lưu trữ dữ liệu liên tục trong bộ nhớ C (contiguous memory) và tận dụng chỉ thị phần cứng SIMD (Single Instruction, Multiple Data), NumPy tăng tốc độ xử lý lên gấp **50 đến 100 lần** so với Python thuần.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Python List vs NumPy ndarray
* **Python List:** Là một mảng chứa các con trỏ (pointers) trỏ tới các đối tượng Python phân tán rải rác trong bộ nhớ Heap. Mỗi phần tử phải mang theo metadata kiểu dữ liệu (overhead lớn, cache miss liên tục).
* **NumPy ndarray:** Là một khối bộ nhớ liền kề (contiguous block of memory) chứa các phần tử đồng nhất kiểu (homogeneous data type, ví dụ `float32` hoặc `int64`). Không tốn chi phí kiểm tra kiểu động tại runtime.

### 2. Nguyên lý Vectorization & Broadcasting
* **Vectorization:** Thay vì viết vòng lặp duyệt qua từng phần tử:
  ```python
  # Cach cu (Slow)
  c = [a[i] + b[i] for i in range(len(a))]
  # Vectorized (Fast - chay o tang C)
  c = a + b
  ```
* **Broadcasting Rules:** Cho phép thực hiện phép toán số học giữa hai mảng có kích thước khác nhau nếu:
  1. Hai chiều bằng nhau, HOẶC
  2. Một trong hai chiều có kích thước bằng 1.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Broadcasting Intuition:
Ma trận A (3x3):         Vector B (1x3):       Kết quả A + B (3x3):
[[1, 2, 3],             [[10, 20, 30]]        [[11, 22, 33],
 [4, 5, 6],         +   (Tự nhân bản       =   [14, 25, 36],
 [7, 8, 9]]              thành 3 dòng)         [17, 28, 39]]
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import numpy as np
import time

# 1. So sanh hieu nang: Python List vs NumPy Array tren 1,000,000 phan tu
N = 1_000_000
list_a = list(range(N))
list_b = list(range(N))

arr_a = np.arange(N, dtype=np.float64)
arr_b = np.arange(N, dtype=np.float64)

# Do thoi gian Python list
start_time = time.time()
list_c = [list_a[i] + list_b[i] for i in range(N)]
python_time = time.time() - start_time

# Do thoi gian NumPy Vectorization
start_time = time.time()
arr_c = arr_a + arr_b
numpy_time = time.time() - start_time

speedup = python_time / numpy_time
print(f"Thoi gian Python List : {python_time:.5f} giay")
print(f"Thoi gian NumPy Array : {numpy_time:.5f} giay")
print(f"-> NumPy nhanh hon gap : {speedup:.1f} lan!")

### 🔍 Code Walkthrough (Khối 1)
Khối code đo lường thời gian thực thi phép cộng 1 triệu phần tử giữa danh sách Python thuần và mảng NumPy.
- **Input:** 2 mảng $10^6$ số nguyên.
- **Output:** Thời gian thực thi và hệ số tăng tốc (Speedup factor).
- **Tại sao viết như vậy:** Chứng minh bằng thực nghiệm trực quan rằng Vectorization không chỉ là thói quen viết code gọn, mà là bắt buộc để sống sót trong xử lý dữ liệu lớn.


In [ ]:
# 2. Thao tac mang da chieu, Broadcasting va Boolean Masking
# Tao ma tran diem thi cua 5 thi sinh qua 3 bai thi Olympic
scores = np.array([
    [75, 82, 90],
    [60, 65, 70],
    [95, 88, 92],
    [50, 45, 55],
    [85, 90, 88]
])

print("Ma tran diem goc (Shape: 5x3):\n", scores)

# Tinh diem trung binh cua tung thi sinh (tinh theo truc ngang - axis=1)
student_means = np.mean(scores, axis=1, keepdims=True)
print("\nDiem trung binh tung thi sinh:\n", student_means)

# Broadcasting: Chuan hoa diem bang cach tru di diem trung binh cua tung bai (axis=0)
exam_means = np.mean(scores, axis=0)
centered_scores = scores - exam_means
print("\nDiem da tru trung vi mon thi (Centered):\n", centered_scores)

# Boolean Masking: Loc ra tat ca cac diem >= 85
high_scores = scores[scores >= 85]
print("\nCac diem dat loai Gioi (>= 85):", high_scores)

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa các thao tác cốt lõi trên ma trận:
- `axis=1` (tính dọc theo cột để thu về trung bình từng dòng) và `axis=0` (tính dọc theo dòng để thu về trung bình từng cột).
- `keepdims=True` giữ nguyên số chiều 2D để sẵn sàng cho Broadcasting.
- **Boolean Masking:** `scores[scores >= 85]` lọc trực tiếp các phần tử thỏa mãn điều kiện mà không cần duyệt vòng lặp `for`.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Tạo mảng và biến đổi chiều
Tạo một mảng NumPy 1D gồm 24 số từ 1 đến 24 (`np.arange(1, 25)`), sau đó biến đổi thành ma trận 2D có kích thước `(4, 6)` và ma trận 3D có kích thước `(2, 3, 4)`.

### Bài 2 (Easy): Chuẩn hóa Min-Max bằng Vectorization
Cho một mảng 1D `x = np.array([12, 45, 67, 89, 34, 23, 90, 11])`. Hãy viết công thức vectơ hóa tính Chuẩn hóa Min-Max:
$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$
Đảm bảo kết quả nằm trong đoạn $[0, 1]$.

### Bài 3 (Medium): Thay thế phần tử theo điều kiện
Cho một ma trận dữ liệu số học ngẫu nhiên `data = np.random.randn(5, 5)`. Hãy thay thế tất cả các giá trị âm (`< 0`) bằng số `0` chỉ bằng một dòng lệnh duy nhất (tương đương hàm kích hoạt ReLU).

### Bài 4 (Medium): Tính khoảng cách Euclid giữa các điểm dữ liệu
Cho tập 3 điểm trong không gian 2D `points = np.array([[0, 0], [3, 4], [1, 1]])` và điểm truy vấn `query = np.array([1, 2])`. Dùng Broadcasting để tính khoảng cách Euclid từ điểm truy vấn tới tất cả các điểm trong `points`.

### Bài 5 (Challenge): Tối ưu hóa phép nhân ma trận thủ công
Viết phép nhân ma trận giữa hai ma trận $A (N \times K)$ và $B (K \times M)$ bằng Python loops, so sánh thời gian chạy với toán tử `@` (`np.dot(A, B)`) khi $N=500, K=500, M=500$.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phần thi Competitive AI, khi cần tạo ra các đặc trưng tương tác (interaction features) như tỷ lệ giữa hai cột hay khoảng cách không gian giữa hàng nghìn tọa độ, việc viết vòng lặp `for` sẽ khiến notebook của bạn vượt ngưỡng giới hạn thời gian chạy (timeout). Làm chủ NumPy Vectorization là kỹ năng sống còn giúp bạn xử lý các phép biến đổi dữ liệu tức thời trong chớp mắt.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một thí sinh cố gắng cộng một vector trọng số vào ma trận dữ liệu nhưng bị báo lỗi `ValueError: operands could not be broadcast together with shapes (4, 3) (4,)`:

```python
# CODE CÓ LỖI BROADCASTING:
X = np.ones((4, 3))   # 4 dong, 3 cot
bias = np.array([1, 2, 3, 4]) # 4 phan tu

# Thi sinh muon cong bias vao tung cot (moi dong duoc cong bias tuong ung):
# result = X + bias # Gay ra loi!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao `bias` có shape `(4,)` không thể broadcast tự động với `(4, 3)`.
2. Sửa lỗi bằng cách sử dụng `np.reshape`, `bias[:, np.newaxis]` hoặc `.reshape(4, 1)` để thực hiện phép cộng chính xác.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Lý do cơ bản nhất giúp NumPy nhanh hơn nhiều lần so với Python List thuần là gì?
   - A. NumPy chạy trên card đồ họa GPU theo mặc định.
   - B. NumPy lưu trữ dữ liệu liên tục trong bộ nhớ, kiểu dữ liệu đồng nhất và thực thi ở tầng ngôn ngữ C tối ưu.
   - C. NumPy bỏ qua việc kiểm tra độ chính xác số học.
   - D. NumPy tự động xóa các biến rác trong máy tính.

2. Phép toán `np.mean(matrix, axis=0)` trên một ma trận 2D có kích thước $(10, 5)$ sẽ cho ra kết quả có kích thước nào?
   - A. $(10, 1)$
   - B. Một số vô hướng duy nhất
   - C. $(5,)$
   - D. $(10, 5)$

3. Trong NumPy, biểu thức `arr[arr % 2 == 0]` được gọi là kỹ thuật gì?
   - A. Integer Fancy Indexing
   - B. Slicing đa chiều
   - C. Boolean Masking
   - D. Data Imputation

4. Hai mảng có kích thước $(8, 1, 6, 1)$ và $(7, 1, 5)$ có thể broadcast với nhau được không?
   - A. Có thể, vì kích thước tổng bằng nhau.
   - B. Có thể, theo quy tắc căn lề từ phải qua trái.
   - C. Không thể, vì các chiều tương ứng không bằng nhau và không có chiều nào bằng 1.
   - D. Chỉ broadcast được nếu mảng có kiểu float32.

5. Cú pháp nào là chuẩn mực trong Python hiện đại để thực hiện phép nhân ma trận giữa hai mảng 2D $A$ và $B$?
   - A. `A * B`
   - B. `A @ B`
   - C. `A.multiply(B)`
   - D. `A ^ B`



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Bản chất tối ưu hóa bộ nhớ của NumPy so với Python List.
  - Sức mạnh của Vectorization và luật Broadcasting đa chiều.
  - Cách dùng Boolean Masking để trích xuất và lọc dữ liệu tốc độ cao.
* **What I can now do:**
  - Viết code xử lý mảng số học chuẩn C mà không cần viết vòng lặp `for`.
  - Tự tin thực hiện các phép biến đổi ma trận, tính toán thống kê theo trục `axis=0` và `axis=1`.
* **What I still need to practice:**
  - Chuyển giao tư duy mảng số học sang cấu trúc bảng dữ liệu có nhãn cột trong **Pandas**.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M1_S02**, chúng ta sẽ tiến lên một bậc: Sử dụng **Pandas Core** để nạp dữ liệu từ các file thực tế (CSV/Excel), làm quen với Series & DataFrame, và làm chủ nghệ thuật truy vấn lọc dữ liệu phức tạp (`loc`, `iloc`, boolean indexing).



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M1**:
- Python dành cho data & cấu trúc bộ nhớ ✓
- NumPy basics & Vectorization ✓
- Broadcasting & Boolean Masking ✓

* **Session Deliverable:** Mã nguồn tính toán số học tốc độ cao trên mảng đa chiều
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
